# 03. Test и итоговые сравнения
T4 нужна для генерации. Укажи те же имя и признак, что в 01. Настройки и версия кода уже сохранены в запуске.

In [ ]:
from pathlib import Path
from runpy import run_path
from subprocess import run

REPO = "https://github.com/danilov-e/lora-activation-steering.git"
RUN_NAME = "research_api_01"
TRAIT = "sycophancy"  # evil / sycophancy / hallucination
GIT_REF = "main"      # используется при создании нового запуска

SOURCE = Path("/content/lora-steering-bootstrap")
if not SOURCE.exists():
    run(["git", "clone", "--quiet", "--depth", "1", REPO, str(SOURCE)], check=True)

colab = run_path(str(SOURCE / "notebooks" / "colab.py"))
PROJECT, RUNS, COMMIT = colab["setup"](SOURCE, RUN_NAME, TRAIT, GIT_REF, resume=True)

from src import judge as j, report as r, workflow as w
from IPython.display import Image, display
import pandas as pd

In [ ]:
RUN = w.open_run(PROJECT, RUN_NAME, TRAIT, runs_dir=RUNS)
CONFIG = w.run_info(RUN)["config"]
choices = w.load_choices(RUN)
print(RUN)
display(pd.DataFrame(choices))

TEST = RUN / "test" / "all"
expected = (2 + 8 * len(choices)) * len(w.pairs(RUN, "test")) * CONFIG["test_repeats"]
batch_size = CONFIG["judge"]["batch_size"]
print(f"Test: {expected} ответов, около {(expected + batch_size - 1) // batch_size} запросов")

## Генерация
До генерации проверяются итоговые веса и векторы всех адаптеров. При повторе используются сохранённые ответы.

In [ ]:
w.generate_test(RUN)

## Оценка через OpenAI
Следующая ячейка запускает платные вызовы. Судья и параметры берутся из сохранённого запуска.

In [ ]:
colab["api_key"]()
display(j.judge(TEST, TRAIT, **CONFIG["judge"]))

## Итоговые эффекты
Сравниваем прирост признака относительно собственного baseline и разрыв относительно базовой модели под исходным вектором. Интервалы считаются по вопросам, после усреднения повторов.

In [ ]:
frame = r.scored(TEST, TRAIT)
summary, effects, figures = r.test_report(frame, RUN)
display(summary.round(2))
display(effects.round(2))
for plot in figures:
    display(Image(filename=str(plot)))

In [ ]:
for choice in choices:
    name = choice["name"]
    print(name)
    display(r.examples(frame, TRAIT, f"steered_{name}:{name}_old"))